In [22]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from lime.lime_tabular import LimeTabularExplainer
from tqdm import tqdm

ind_hosp = pd.read_parquet("ind_hosp.parquet")
calibrated = joblib.load("lightgbm.pkl")
base_model = calibrated.calibrated_classifiers_[0].estimator

cols_to_drop = [
    "subject_id",
    "hadm_id",
    "dischtime",
    "current_date",
    "target_readmission_30d",
    "los"
]

X = ind_hosp.drop(columns=cols_to_drop)
y = ind_hosp["target_readmission_30d"]

bool_cols = X.select_dtypes(include="bool").columns
X[bool_cols] = X[bool_cols].astype(int)

patient_target = ind_hosp.groupby('subject_id')['target_readmission_30d'].max().reset_index()
patient_target.columns = ['subject_id', 'has_readmission']

train_val_ids, test_ids = train_test_split(
    patient_target['subject_id'],
    test_size=0.2,
    random_state=42,
    stratify=patient_target['has_readmission']
)

train_ids, val_ids = train_test_split(
    train_val_ids,
    test_size=0.125,
    random_state=42,
    stratify=patient_target[patient_target['subject_id'].isin(train_val_ids)]['has_readmission']
)

train_mask = ind_hosp['subject_id'].isin(train_ids)
val_mask = ind_hosp['subject_id'].isin(val_ids)
test_mask = ind_hosp['subject_id'].isin(test_ids)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

In [23]:
print(f"Rows: {len(X_test)}")
print(f"Hospitalizations: {ind_hosp.loc[test_mask, 'hadm_id'].nunique()}")
print(f"Patients: {ind_hosp.loc[test_mask, 'subject_id'].nunique()}")

Rows: 365442
Hospitalizations: 81616
Patients: 35915


In [24]:
X_train_lime = X_train.copy()
X_test_lime = X_test.copy()

icd_cols = [c for c in X_train.columns if c.startswith("icd_")]
ccsr_cols = [c for c in X_train.columns if c.startswith("ccsr_")]

X_train_lime[icd_cols] = X_train_lime[icd_cols].fillna(0)
X_test_lime[icd_cols] = X_test_lime[icd_cols].fillna(0)

In [25]:
clinical_ranges = {
    'lab_50983_daily': {'low': 135, 'high': 145}, #sodium
    'lab_50971_daily': {'low': 3.5, 'high': 5}, #potassium
    'lab_50902_daily': {'low': 95, 'high': 105}, #chloride
    'lab_50882_daily': {'low': 22, 'high': 32}, #bicarbonate
    'lab_50912_daily': {'low': 0.8, 'high': 1.3}, #creatinine
    'lab_51006_daily': {'low': 8, 'high': 21}, #BUN
    'lab_50931_daily': {'low': 65, 'high': 110}, #glucose
    'lab_50893_daily': {'low': 8.5, 'high': 10.2}, #calcium
    'lab_50868_daily': {'low': 10, 'high': 18}, #anion gap
    'lab_51222_daily': {
        'male': {'low': 13, 'high': 17},
        'female': {'low': 12, 'high': 15}
    }, #hemoglobin
    'lab_51301_daily': {'low': 4, 'high': 10}, #WBC
    'lab_51265_daily': {'low': 150, 'high': 400}, #Platelet Count
    'lab_51221_daily': {
        'male': {'low': 40, 'high': 52},
        'female': {'low': 36, 'high': 47}
    }, #Hematocrit
    'lab_51250_daily': {'low': 80, 'high': 100}, #MCV
    'lab_51277_daily': {'low': 11.5, 'high': 14.5}, #RDW
    'lab_50960_daily': {'low': 1.5, 'high': 2}, #magnesium
    'lab_50970_daily': {'low': 2.7, 'high': 4.5}, #phosphate
    'lab_51248_daily': {'low': 26, 'high': 32}, #MCH
    'lab_51249_daily': {'low': 30, 'high': 35}, #MCHC
    'lab_51279_daily': {
        'male': {'low': 4.5, 'high': 5.9},
        'female': {'low': 4, 'high': 5.2}
    } #RBC
}

def normal_lab_value(col, gender=None):
    ref = clinical_ranges[col]
    if "male" in ref:
        if gender == 1:
            low = ref["male"]["low"]
            high = ref["male"]["high"]
        else:
            low = ref["female"]["low"]
            high = ref["female"]["high"]
    else:
        low = ref["low"]
        high = ref["high"]

    return (low + high) / 2

In [26]:
lab_cols = [c for c in X_train.columns if c.startswith("lab_")]

for col in lab_cols:
    if col not in clinical_ranges:
        continue

    missing = X_train_lime[col].isna()
    male_idx = missing & (X_train_lime["gender_male"] == 1)
    female_idx = missing & (X_train_lime["gender_male"] == 0)
    X_train_lime.loc[male_idx, col] = normal_lab_value(col, 1)
    X_train_lime.loc[female_idx, col] = normal_lab_value(col, 0)

for col in lab_cols:
    if col not in clinical_ranges:
        continue

    missing = X_test_lime[col].isna()
    male_idx = missing & (X_test_lime["gender_male"] == 1)
    female_idx = missing & (X_test_lime["gender_male"] == 0)
    X_test_lime.loc[male_idx, col] = normal_lab_value(col, 1)
    X_test_lime.loc[female_idx, col] = normal_lab_value(col, 0)

In [27]:
#X_test_lime = X_test_lime[:100]
icd_cols = [col for col in X.columns if col.startswith('icd_')]
ccsr_cols = [col for col in X.columns if col.startswith('ccsr_')]
lab_cols = [col for col in X.columns if col.startswith('lab_') and col.endswith('_daily')]

features_to_analyze = (
    icd_cols +
    ccsr_cols +
    lab_cols +
    [
        'num_diagnoses',
        'num_chronic',
        'comorbidity_score',
        'num_medications_daily',
        'prior_admissions_12mo',
        'cumulative_procedures',
        'cumulative_medications',
        'num_procedures_daily',
        'gender_male',
        'age'
    ]
)

explainer = LimeTabularExplainer(
    training_data=X_train_lime.values,
    feature_names=X_train_lime.columns.tolist(),
    class_names=["No readmission", "Readmission"],
    mode="classification",
    discretize_continuous=False,
    random_state=42
)

predict_fn = lambda x: calibrated.predict_proba(pd.DataFrame(x, columns=X_train_lime.columns))
all_results = []
for idx in tqdm(X_test_lime.index,
                total=len(X_test_lime),
                desc="LIME"):

    row = X_test_lime.loc[idx]
    exp = explainer.explain_instance(
        row.values,
        predict_fn,
        num_features=len(X_train_lime.columns),
        num_samples=1000
    )

    weights = {
        X_train_lime.columns[i]: w
        for i, w in exp.local_exp[1]
    }

    pred = predict_fn(row.values.reshape(1, -1))[0, 1]
    for feature in features_to_analyze:

        all_results.append({
            "subject_id": ind_hosp.loc[idx, "subject_id"],
            "hadm_id": ind_hosp.loc[idx, "hadm_id"],
            "feature": feature,
            "lime": weights.get(feature, 0.0),
            "value": row[feature],
            "predicted_proba": pred,
        })

lime_df = pd.DataFrame(all_results)

LIME: 100%|██████████| 365442/365442 [1:25:49<00:00, 70.97it/s]


In [28]:
lime_by_hadm = (
    lime_df
    .groupby(["subject_id", "hadm_id", "feature"])
    .agg(
        lime_mean=("lime", "mean"),
        lime_mean_abs=("lime", lambda x: np.mean(np.abs(x))),
        value_mean=("value", "mean"),
        predicted_proba_mean=("predicted_proba", "mean")
    )
    .reset_index()
)

lime_summary = (
    lime_by_hadm
    .groupby("feature")
    .agg(
        lime_mean=("lime_mean", "mean"),
        lime_mean_abs=("lime_mean_abs", "mean"),
        value_mean=("value_mean", "mean")
    )
    .reset_index()
)

In [29]:
lime_by_hadm.to_csv("lime_by_hadm.csv", index=False)
lime_summary.to_csv("lime_summary.csv", index=False)